# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [87]:
import sys
print(sys.executable)

/Users/ethanjia/ai/projects/tinyml-arduino/bin/python


In [88]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [89]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [90]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [91]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [92]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
X = df[feature_names].values
y = df["Class"].values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", np.unique(y))

X shape: (178, 13)
y shape: (178,)
Classes: [0 1 2]


In [93]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (124, 13)
X_test shape: (54, 13)
y_train shape: (124,)
y_test shape: (54,)


In [94]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("X_train scaled shape:", X_train.shape)
print("X_test scaled shape:", X_test.shape)

X_train scaled shape: (124, 13)
X_test scaled shape: (54, 13)


In [95]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)

y_train_cat shape: (124, 3)
y_test_cat shape: (54, 3)


In [96]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_21 (Dense)            (None, 64)                896       
                                                                 
 dense_22 (Dense)            (None, 32)                2080      
                                                                 
 dense_23 (Dense)            (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [97]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
13/13 [==============================] - 0s 4ms/step - loss: 1.1379 - accuracy: 0.5354 - val_loss: 0.9629 - val_accuracy: 0.6400
Epoch 2/20
13/13 [==============================] - 0s 1ms/step - loss: 0.8122 - accuracy: 0.7273 - val_loss: 0.6897 - val_accuracy: 0.8400
Epoch 3/20
13/13 [==============================] - 0s 1ms/step - loss: 0.5913 - accuracy: 0.8990 - val_loss: 0.4741 - val_accuracy: 0.9600
Epoch 4/20
13/13 [==============================] - 0s 1ms/step - loss: 0.4245 - accuracy: 0.9596 - val_loss: 0.3222 - val_accuracy: 0.9600
Epoch 5/20
13/13 [==============================] - 0s 1ms/step - loss: 0.2980 - accuracy: 0.9798 - val_loss: 0.2184 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 1ms/step - loss: 0.2111 - accuracy: 0.9899 - val_loss: 0.1556 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1549 - accuracy: 0.9899 - val_loss: 0.1168 - val_accuracy: 0.9600
Epoch 8/20
13/13 [==

In [98]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
train_loss, train_acc = model.evaluate(X_train, y_train_cat, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)

print("Training Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Training Accuracy: 0.9919354915618896
Test Accuracy: 0.9629629850387573
2/2 [==============================] - 0s 880us/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       0.95      0.95      0.95        21
           2       0.93      0.93      0.93        15

    accuracy                           0.96        54
   macro avg       0.96      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion Matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  1 14]]


In [99]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

base_model_size_kb = len(tflite_model) / 1024

print("Base Float32 TFLite Model Size: {:.2f} KB".format(base_model_size_kb))

INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpfph5xrxl/assets


INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpfph5xrxl/assets


Base Float32 TFLite Model Size: 14.14 KB


2026-05-19 23:00:17.244106: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:00:17.244116: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:00:17.244208: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpfph5xrxl
2026-05-19 23:00:17.244641: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:00:17.244645: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpfph5xrxl
2026-05-19 23:00:17.245811: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 23:00:17.261464: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpfph5xrxl
2026-05-

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [100]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    # <-- Enter your code here <--#
    tflite_model = converter.convert()

    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_index = input_details[0]["index"]
    output_index = output_details[0]["index"]

    y_pred = []

    for i in range(len(X_test)):
        input_data = X_test[i:i+1].astype(np.float32)

        # If input is quantized, convert float input to int8 using scale and zero point
        if input_details[0]["dtype"] == np.int8:
            input_scale, input_zero_point = input_details[0]["quantization"]
            input_data = input_data / input_scale + input_zero_point
            input_data = np.clip(input_data, -128, 127).astype(np.int8)
        else:
            input_data = input_data.astype(input_details[0]["dtype"])

        interpreter.set_tensor(input_index, input_data)
        interpreter.invoke()

        output_data = interpreter.get_tensor(output_index)

        # If output is quantized, dequantize it back to float
        if output_details[0]["dtype"] == np.int8:
            output_scale, output_zero_point = output_details[0]["quantization"]
            output_data = output_scale * (output_data.astype(np.float32) - output_zero_point)

        predicted_class = np.argmax(output_data, axis=1)[0]
        y_pred.append(predicted_class)

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    with open(filename, "rb") as f:
        model_size_kb = len(f.read()) / 1024

    print(f"\n{quant_type.upper()} TFLite model size: {model_size_kb:.2f} KB")

    print("Accuracy:", np.mean(y_true == y_pred))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


In [101]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
quantize_and_evaluate(
    model=model,
    X_test=X_test,
    y_test_cat=y_test_cat,
    quant_type='int8',
    filename='model_int8.tflite'
)

quantize_and_evaluate(
    model=model,
    X_test=X_test,
    y_test_cat=y_test_cat,
    quant_type='float16',
    filename='model_float16.tflite'
)

quantize_and_evaluate(
    model=model,
    X_test=X_test,
    y_test_cat=y_test_cat,
    quant_type='dynamic',
    filename='model_dynamic.tflite'
)

INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpwhfcrupd/assets


INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpwhfcrupd/assets



INT8 TFLite model size: 5.82 KB
Accuracy: 0.9814814814814815

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       0.95      1.00      0.98        21
           2       1.00      0.93      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix:
[[18  0  0]
 [ 0 21  0]
 [ 0  1 14]]


/Users/ethanjia/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-19 23:00:19.588658: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:00:19.588667: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:00:19.588757: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpwhfcrupd
2026-05-19 23:00:19.589211: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:00:19.589215: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpwhfcrupd
2026-05-19 23:00:19.590338: I tensorflow/cc/saved_model/loader.cc:233]

INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmphm4gzc0_/assets


...

2026-05-19 23:00:19.809587: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmphm4gzc0_
2026-05-19 23:00:19.809998: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:00:19.810002: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmphm4gzc0_
2026-05-19 23:00:19.811065: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 23:00:19.826669: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmphm4gzc0_
2026-05-19 23:00:19.831053: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 21466 microseconds.


INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpprwuno6c/assets


INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpprwuno6c/assets



DYNAMIC TFLite model size: 8.24 KB
Accuracy: 0.9629629629629629

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       0.95      0.95      0.95        21
           2       0.93      0.93      0.93        15

    accuracy                           0.96        54
   macro avg       0.96      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion Matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  1 14]]


2026-05-19 23:00:20.029129: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:00:20.029139: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:00:20.029229: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpprwuno6c
2026-05-19 23:00:20.029644: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:00:20.029648: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpprwuno6c
2026-05-19 23:00:20.030683: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 23:00:20.046275: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpprwuno6c
2026-05-

## Problem 1 - Part (c)

### Pruning

In [102]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
batch_size = 8
epochs = 10

end_step = np.ceil(len(X_train) / batch_size).astype(np.int32) * epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

print("End step:", end_step)

End step: 160


In [103]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential([
    prune_low_magnitude(
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

pruned_model.summary()

Model: "sequential_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 24 (PruneLowMagnitude)                                          
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 25 (PruneLowMagnitude)                                          
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 26 (PruneLowMagnitude)                                          
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [104]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

history_pruned = pruned_model.fit(
    X_train,
    y_train_cat,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 0s 5ms/step - loss: 1.0627 - accuracy: 0.4545 - val_loss: 0.9442 - val_accuracy: 0.6400
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7744 - accuracy: 0.8990 - val_loss: 0.7255 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5913 - accuracy: 0.9596 - val_loss: 0.5619 - val_accuracy: 0.9600
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.4564 - accuracy: 0.9697 - val_loss: 0.4276 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.3473 - accuracy: 0.9697 - val_loss: 0.3184 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.2605 - accuracy: 0.9798 - val_loss: 0.2457 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.1925 - accuracy: 0.9899 - val_loss: 0.1937 - val_accuracy: 0.9600
Epoch 8/10
13/13 [==

In [105]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)
stripped_pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
pruned_tflite_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(pruned_tflite_model)

with open("model_pruned.tflite", "rb") as f:
    pruned_model_size_kb = len(f.read()) / 1024

print("Pruned TFLite Model Size: {:.2f} KB".format(pruned_model_size_kb))

pruned_loss, pruned_acc = stripped_pruned_model.evaluate(X_test, y_test_cat, verbose=0)
print("Pruned Test Accuracy:", pruned_acc)

y_pred_probs = stripped_pruned_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmp7yo8h5sc/assets


INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmp7yo8h5sc/assets


Pruned TFLite Model Size: 14.18 KB
Pruned Test Accuracy: 0.9629629850387573
2/2 [==============================] - 0s 840us/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.90      0.95        21
           2       0.88      1.00      0.94        15

    accuracy                           0.96        54
   macro avg       0.96      0.97      0.96        54
weighted avg       0.97      0.96      0.96        54

Confusion Matrix:
[[18  0  0]
 [ 0 19  2]
 [ 0  0 15]]


2026-05-19 23:01:25.038868: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:01:25.038879: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:01:25.038965: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmp7yo8h5sc
2026-05-19 23:01:25.039224: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:01:25.039228: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmp7yo8h5sc
2026-05-19 23:01:25.039914: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 23:01:25.046413: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmp7yo8h5sc
2026-05-

In [106]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

pruned_loss, pruned_acc = stripped_pruned_model.evaluate(X_test, y_test_cat, verbose=0)

print("Pruned Test Accuracy:", pruned_acc)

y_pred_probs = stripped_pruned_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = np.argmax(y_test_cat, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Pruned Test Accuracy: 0.9629629850387573
2/2 [==============================] - 0s 1ms/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.90      0.95        21
           2       0.88      1.00      0.94        15

    accuracy                           0.96        54
   macro avg       0.96      0.97      0.96        54
weighted avg       0.97      0.96      0.96        54

Confusion Matrix:
[[18  0  0]
 [ 0 19  2]
 [ 0  0 15]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [107]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
student_model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

student_model.summary()

Model: "sequential_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_27 (Dense)            (None, 32)                448       
                                                                 
 dense_28 (Dense)            (None, 16)                528       
                                                                 
 dense_29 (Dense)            (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [110]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
teacher_preds_soft = model.predict(X_train)

print("teacher_preds_soft shape:", teacher_preds_soft.shape)
print("y_train_cat shape:", y_train_cat.shape)
print("First teacher soft label:", teacher_preds_soft[0])

4/4 [==============================] - 0s 829us/step
teacher_preds_soft shape: (124, 3)
y_train_cat shape: (124, 3)
First teacher soft label: [9.9953401e-01 3.8456937e-04 8.1323393e-05]


In [113]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)

def distillation_loss(y_true_combined, y_pred):
    alpha = 0.5

    num_classes = y_train_cat.shape[1]

    hard_labels = y_true_combined[:, :num_classes]
    soft_labels = y_true_combined[:, num_classes:]

    hard_loss = tf.keras.losses.categorical_crossentropy(hard_labels, y_pred)
    soft_loss = tf.keras.losses.KLDivergence()(soft_labels, y_pred)

    return alpha * hard_loss + (1 - alpha) * soft_loss

print("y_train_combined shape:", y_train_combined.shape)
print("distillation_loss is defined.")

y_train_combined shape: (124, 6)
distillation_loss is defined.


In [114]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

history_student = student_model.fit(
    X_train,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 0s 5ms/step - loss: 1.1955 - accuracy: 0.3434 - val_loss: 1.2270 - val_accuracy: 0.2800
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 1.0001 - accuracy: 0.4444 - val_loss: 1.0533 - val_accuracy: 0.4000
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.8600 - accuracy: 0.6061 - val_loss: 0.9072 - val_accuracy: 0.5600
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7404 - accuracy: 0.8283 - val_loss: 0.7881 - val_accuracy: 0.7600
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.6349 - accuracy: 0.9192 - val_loss: 0.6715 - val_accuracy: 0.8800
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5401 - accuracy: 0.9394 - val_loss: 0.5558 - val_accuracy: 0.8800
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.4504 - accuracy: 0.9697 - val_loss: 0.4566 - val_accuracy: 0.9600
Epoch 8/10
13/13 [==

In [115]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
student_tflite_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(student_tflite_model)

with open("model_kd.tflite", "rb") as f:
    student_model_size_kb = len(f.read()) / 1024

print("Knowledge Distillation TFLite Model Size: {:.2f} KB".format(student_model_size_kb))

INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpj3m_hj75/assets


INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpj3m_hj75/assets


Knowledge Distillation TFLite Model Size: 6.14 KB


2026-05-19 23:03:20.840254: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:03:20.840264: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:03:20.840356: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpj3m_hj75
2026-05-19 23:03:20.840811: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:03:20.840815: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpj3m_hj75
2026-05-19 23:03:20.841926: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 23:03:20.858221: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpj3m_hj75
2026-05-

In [116]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred_probs = student_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = np.argmax(y_test_cat, axis=1)

print("Knowledge Distillation Test Accuracy:", np.mean(y_true == y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 1ms/step
Knowledge Distillation Test Accuracy: 0.9074074074074074

Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95        18
           1       1.00      0.76      0.86        21
           2       0.83      1.00      0.91        15

    accuracy                           0.91        54
   macro avg       0.91      0.92      0.91        54
weighted avg       0.92      0.91      0.90        54

Confusion Matrix:
[[18  0  0]
 [ 2 16  3]
 [ 0  0 15]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [118]:
# <-- (if needed) Enter your code here <--#
# Problem 1 - Part (e)
# Further reduce model size by applying TFLite dynamic range quantization
# to the knowledge-distilled student model.

import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

# 1. Convert the trained student model to normal TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
student_tflite_model = converter.convert()

student_tflite_filename = "student_model.tflite"
with open(student_tflite_filename, "wb") as f:
    f.write(student_tflite_model)

student_tflite_size_kb = os.path.getsize(student_tflite_filename) / 1024
print("Student TFLite Model Size: {:.2f} KB".format(student_tflite_size_kb))


# 2. Convert the student model to quantized TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

quant_student_tflite_model = converter.convert()

quant_student_tflite_filename = "quantized_student_model.tflite"
with open(quant_student_tflite_filename, "wb") as f:
    f.write(quant_student_tflite_model)

quant_student_tflite_size_kb = os.path.getsize(quant_student_tflite_filename) / 1024
print("Quantized Student TFLite Model Size: {:.2f} KB".format(quant_student_tflite_size_kb))


# 3. Evaluate the quantized TFLite model on the test set
interpreter = tf.lite.Interpreter(model_path=quant_student_tflite_filename)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

y_pred_classes = []

for i in range(len(X_test)):
    input_data = X_test[i:i+1].astype(np.float32)

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()

    output_data = interpreter.get_tensor(output_details[0]['index'])
    pred_class = np.argmax(output_data, axis=1)[0]
    y_pred_classes.append(pred_class)

y_pred_classes = np.array(y_pred_classes)
y_true_classes = np.argmax(y_test_cat, axis=1)

quant_student_acc = np.mean(y_pred_classes == y_true_classes)

print("Quantized Student Test Accuracy:", quant_student_acc)
print("\nClassification Report:")
print(classification_report(y_true_classes, y_pred_classes))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_classes, y_pred_classes))
reduction_percent = (1 - quant_student_tflite_size_kb / student_tflite_size_kb) * 100
print("Size reduction from student TFLite: {:.2f}%".format(reduction_percent))

INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmphpxb2j3a/assets


INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmphpxb2j3a/assets


Student TFLite Model Size: 6.14 KB


2026-05-20 10:28:06.591223: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 10:28:06.591234: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 10:28:06.591325: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmphpxb2j3a
2026-05-20 10:28:06.591733: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 10:28:06.591737: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmphpxb2j3a
2026-05-20 10:28:06.592800: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 10:28:06.608438: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmphpxb2j3a
2026-05-

INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpea5293kd/assets


INFO:tensorflow:Assets written to: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpea5293kd/assets


Quantized Student TFLite Model Size: 6.14 KB
Quantized Student Test Accuracy: 0.9074074074074074

Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95        18
           1       1.00      0.76      0.86        21
           2       0.83      1.00      0.91        15

    accuracy                           0.91        54
   macro avg       0.91      0.92      0.91        54
weighted avg       0.92      0.91      0.90        54


Confusion Matrix:
[[18  0  0]
 [ 2 16  3]
 [ 0  0 15]]
Size reduction from student TFLite: -0.06%


2026-05-20 10:28:06.810867: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 10:28:06.810878: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 10:28:06.810977: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpea5293kd
2026-05-20 10:28:06.811383: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 10:28:06.811388: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpea5293kd
2026-05-20 10:28:06.812451: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 10:28:06.828235: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/h8/_8jg9xyj0b1fv064kh64kr300000gn/T/tmpea5293kd
2026-05-

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
